# Scalpel Tutorial

This notebook demonstrates the core functionality of the **Scalpel** library for FreeSurfer cortical surface analysis.

We'll use the `bert` sample subject that comes with FreeSurfer.

## Contents
1. Subject Initialization & Surface Data
2. Working with Labels
3. Visualization
4. Measurements (Surface Area, Thickness, Depth, Distance)
5. Analysis (Clustering, Thresholding, Gyral-Sulcal Relationships)

## 1. Subject Initialization & Surface Data

In [ ]:
from scalpel.subject import ScalpelSubject
from pathlib import Path
import numpy as np
import os

# Use FreeSurfer's sample subject 'bert'
subjects_dir = Path(os.environ.get('FREESURFER_HOME', '/Users/benparker/freesurfer')) / 'subjects'
print(f"FreeSurfer subjects directory: {subjects_dir}")

In [ ]:
# Initialize a ScalpelSubject with the 'bert' sample brain
# Parameters:
#   - subject_id: FreeSurfer subject name
#   - hemi: hemisphere ('lh' = left, 'rh' = right)
#   - subjects_dir: path to FreeSurfer subjects directory
#   - surface_type: which surface to use ('inflated', 'pial', 'white')

subject = ScalpelSubject(
    subject_id="bert",
    hemi="lh",
    subjects_dir=subjects_dir,
    surface_type="inflated"
)

print(f"Subject ID: {subject.subject_id}")
print(f"Hemisphere: {subject.hemi}")
print(f"Surface type: {subject.surface_type}")
print(f"Subject path: {subject.subject_fs_path}")

### Accessing Surface Geometry

In [ ]:
# Access surface geometry data
vertices = subject.surface_RAS  # RAS coordinates of all vertices
faces = subject.faces           # Triangular faces defining the mesh
vertex_indices = subject.vertex_indexes  # Unique vertex indices

print(f"Number of vertices: {len(vertices):,}")
print(f"Number of faces: {len(faces):,}")
print(f"Vertex coordinate shape: {vertices.shape}")
print(f"\nSample vertices (first 5):")
print(vertices[:5])

### Morphometric Data (Curvature, Thickness, Sulcal Depth)

In [ ]:
# Access morphometric data from FreeSurfer
curvature = subject.mean_curvature  # Mean curvature at each vertex
thickness = subject.thickness        # Cortical thickness at each vertex
sulc_depth = subject.sulc_vals       # Sulcal depth values

print(f"Mean curvature - range: [{curvature.min():.3f}, {curvature.max():.3f}]")
print(f"Cortical thickness - range: [{thickness.min():.3f}, {thickness.max():.3f}] mm")
print(f"Sulcal depth values - range: [{sulc_depth.min():.3f}, {sulc_depth.max():.3f}]")

In [ ]:
# Gyral and sulcal regions are automatically identified based on curvature
# Negative curvature = gyrus (crown), Positive curvature = sulcus (fundus)

gyral_vertices = subject.gyrus   # Vertex indices of gyral regions
sulcal_vertices = subject.sulcus # Vertex indices of sulcal regions

print(f"Number of gyral vertices: {len(gyral_vertices):,}")
print(f"Number of sulcal vertices: {len(sulcal_vertices):,}")
print(f"Gyrus/sulcus ratio: {len(gyral_vertices)/len(sulcal_vertices):.2f}")

## 2. Working with Labels

Labels are regions of interest (ROIs) on the brain surface. They can be loaded from FreeSurfer label files or created programmatically.

In [ ]:
# Load labels from FreeSurfer's automatic parcellation
# These labels are in the subject's label/ directory

# Load the cortex label (entire cortical surface)
subject.load_label('cortex')

print(f"Loaded label: cortex")
print(f"Number of vertices in cortex: {len(subject.labels['cortex'].vertex_indexes):,}")

In [ ]:
# List available labels in the subject's label directory
label_dir = subject.subject_fs_path / 'label'
available_labels = list(label_dir.glob(f"{subject.hemi}.*.label"))

print(f"Available labels for {subject.hemi} hemisphere:")
for label_path in sorted(available_labels)[:15]:  # Show first 15
    label_name = label_path.stem.replace(f"{subject.hemi}.", "")
    print(f"  - {label_name}")
if len(available_labels) > 15:
    print(f"  ... and {len(available_labels) - 15} more")

In [ ]:
# Access data from a loaded label
cortex_label = subject.labels['cortex']

print(f"Label name: cortex")
print(f"Vertex indices shape: {cortex_label.vertex_indexes.shape}")
print(f"RAS coordinates shape: {cortex_label.label_RAS.shape}")
print(f"\nFirst 5 vertex indices: {cortex_label.vertex_indexes[:5]}")
print(f"First 5 RAS coordinates:\n{cortex_label.label_RAS[:5]}")

In [ ]:
# Create a custom label programmatically
# Let's create a label for the 1000 vertices with highest sulcal depth

# Get indices of deepest sulcal vertices
n_vertices = 1000
deepest_indices = np.argsort(subject.sulc_vals)[-n_vertices:]
deepest_coords = subject.surface_RAS[deepest_indices]

# Load as a custom label
subject.load_label(
    'deepest_sulci_custom',
    label_idxs=deepest_indices,
    label_RAS=deepest_coords
)

print(f"Created custom label 'deepest_sulci_custom' with {n_vertices} vertices")
print(f"Loaded labels: {list(subject.labels.keys())}")

## 3. Visualization

Scalpel provides interactive 3D visualization of brain surfaces and labels.

In [ ]:
# Plot the brain surface
# Available views: 'lateral', 'medial', 'dorsal', 'ventral'

subject.plot(view='lateral')

In [ ]:
# Plot our custom deepest sulci label
subject.plot_label('deepest_sulci_custom', view='lateral', face_colors='red')

## 4. Measurements

Scalpel replicates FreeSurfer's anatomical statistics calculations.

Several measurements rely on FreeSurfer derivative files that are **not all produced by a default `recon-all`**. Generate the missing ones per subject first:

| Measurement | Required file(s) | How to generate |
|---|---|---|
| Gray matter volume (whole hemisphere) | `?h.cortex.label` | standard `recon-all` |
| Sulcal depth | `?h.pial-outer-smoothed` | `recon-all -localGI` |
| Absolute Gaussian curvature | `?h.curv.K` | `mris_curvature -k ?h.white` |
| Folding index / intrinsic curvature index | `?h.white.max`, `?h.white.min` | `mris_curvature -min -max ?h.white` |

Each accessor raises a clear `FileNotFoundError` if its file is missing. These calculations are validated against `mris_anatomical_stats` — gray matter volume (TH3) and the folding index match exactly on `fsaverage`.

### Surface Area

In [ ]:
# Calculate surface area for the entire cortex
total_area = subject.calculate_surface_area()
print(f"Total cortical surface area: {total_area:,.2f} mm²")



### Cortical Thickness

In [ ]:
# Calculate mean cortical thickness
mean_thickness, std_thickness = subject.calculate_cortical_thickness()
print(f"Mean cortical thickness: {mean_thickness:.3f} ± {std_thickness:.3f} mm")

# Calculate thickness for the cortex label
cortex_thickness, cortex_std = subject.calculate_cortical_thickness('cortex')
print(f"Cortex label thickness: {cortex_thickness:.3f} ± {cortex_std:.3f} mm")

### Sulcal Depth

In [ ]:
# Calculate sulcal depth for a label
# This computes the distance between pial and gyral-inflated surfaces

depth = subject.calculate_sulcal_depth(
    'deepest_sulci_custom',
    n_deepest=100,      # Use the 100 deepest vertices
    use_n_deepest=True  # Use count instead of percentage
)
print(f"Sulcal depth (deepest 100 vertices): {depth:.2f} mm")

# Using percentage instead
depth_pct = subject.calculate_sulcal_depth(
    'deepest_sulci_custom',
    depth_pct=10,       # Use deepest 10%
    use_n_deepest=False
)
print(f"Sulcal depth (deepest 10%): {depth_pct:.2f} mm")

### Gray Matter Volume

In [ ]:
# Gray matter volume between the white and pial surfaces.
# Uses FreeSurfer's exact TH3 method (per-face tetrahedra volume summed to
# vertices) and matches mris_anatomical_stats. The whole-hemisphere call masks
# to the cortex label (?h.cortex.label), excluding the medial wall.
total_volume = subject.calculate_gray_matter_volume()
print(f"Total gray matter volume: {total_volume:,.2f} mm³")

# Per-label volume sums the TH3 vertex volumes over the label's vertices.
cortex_volume = subject.calculate_gray_matter_volume('cortex')
print(f"Cortex label volume: {cortex_volume:,.2f} mm³")


### Local Gyrification Index

Local gyrification index (lGI; Schaer et al. 2008) quantifies cortical folding as the ratio of buried pial surface area to smooth outer-hull area within a 25 mm disc. Computed from the pial and `?h.pial-outer-smoothed` surfaces (the latter produced by `recon-all -localGI`).

In [ ]:
# Mean lGI over a label (also cached onto the label's measurements)
mean_lgi = subject.calculate_local_gyrification_index('cortex')
print(f"Mean cortical lGI: {mean_lgi:.3f}")

# The full per-vertex lGI map is available as well
lgi_map = subject.pial_lgi
print(f"Per-vertex lGI range: [{lgi_map.min():.2f}, {lgi_map.max():.2f}]")


### Distance Between Labels

Scalpel offers two distance metrics between labels:

- **Euclidean** — straight-line 3D distance (`calculate_euclidean_distance`), methods `centroid` / `nearest` / `farthest`.
- **Geodesic** — exact on-surface distance that follows the folded cortex (`calculate_geodesic_distance`), so labels on opposite banks of a sulcus are not treated as close. Computed on the subject's loaded surface; requires the `pygeodesic` package.


In [ ]:
# Create two labels for distance measurement
# Label 1: Anterior cortex (x > 0)
anterior_mask = subject.surface_RAS[:, 1] > 30  # anterior (positive y)
anterior_indices = np.where(anterior_mask)[0][:500]  # Take 500 vertices
subject.load_label('anterior_region', label_idxs=anterior_indices, 
                   label_RAS=subject.surface_RAS[anterior_indices])

# Label 2: Posterior cortex (x < 0)
posterior_mask = subject.surface_RAS[:, 1] < -30  # posterior (negative y)
posterior_indices = np.where(posterior_mask)[0][:500]
subject.load_label('posterior_region', label_idxs=posterior_indices,
                   label_RAS=subject.surface_RAS[posterior_indices])

print("Created 'anterior_region' and 'posterior_region' labels")

In [ ]:
# Calculate Euclidean distance between labels
# Methods: 'centroid' (between centroids), 'nearest' (closest points), 'farthest' (furthest points)

dist_centroid = subject.calculate_euclidean_distance(
    'anterior_region', 'posterior_region', method='centroid'
)
print(f"Distance between centroids: {dist_centroid:.2f} mm")

dist_nearest = subject.calculate_euclidean_distance(
    'anterior_region', 'posterior_region', method='nearest'
)
print(f"Nearest point distance: {dist_nearest:.2f} mm")

dist_farthest = subject.calculate_euclidean_distance(
    'anterior_region', 'posterior_region', method='farthest'
)
print(f"Farthest point distance: {dist_farthest:.2f} mm")

#### Geodesic (on-surface) distance

In [ ]:
# Geodesic distance follows the cortical surface (exact MMP algorithm, pygeodesic).
# Methods: 'centroid' (between each label's centroid vertex),
#          'nearest'  (minimum geodesic distance between the two label vertex sets).

geo_centroid = subject.calculate_geodesic_distance(
    'anterior_region', 'posterior_region', method='centroid'
)
print(f"Geodesic distance (centroid): {geo_centroid:.2f} mm")

geo_nearest = subject.calculate_geodesic_distance(
    'anterior_region', 'posterior_region', method='nearest'
)
print(f"Geodesic distance (nearest):  {geo_nearest:.2f} mm")

# On a given surface the geodesic distance is >= the Euclidean distance,
# since it cannot cut straight through the cortex.


### All FreeSurfer Statistics

`calculate_all_freesurfer_stats(label)` runs every measurement above for a label and returns them as a dict, replicating `mris_anatomical_stats`. It therefore needs the curvature derivative files listed at the top of this section (`?h.curv.K`, `?h.white.max`, `?h.white.min`).


In [ ]:
# Calculate all FreeSurfer anatomical statistics for a label
# This replicates the output of mris_anatomical_stats
subject.load_label('cortex') 

all_stats = subject.calculate_all_freesurfer_stats('cortex')

print("FreeSurfer Anatomical Statistics for 'cortex' label:")
print("=" * 50)
for stat_name, value in all_stats.items():
    if isinstance(value, float):
        print(f"{stat_name}: {value:,.4f}")
    else:
        print(f"{stat_name}: {value}")

## 5. Analysis

Scalpel provides advanced analysis tools for gyral-sulcal relationships, clustering, and thresholding.

### Gyral Clustering

In [ ]:
# Perform clustering on gyral regions
# Algorithms: 'kmeans', 'agglomerative', 'dbscan'

clusters = subject.perform_gyral_clustering(
    n_clusters=100,      # Number of clusters
    algorithm='kmeans'   # Clustering algorithm
)

print(f"Number of gyral clusters: {len(np.unique(clusters))}")
print(f"Cluster assignments shape: {clusters.shape}")
print(f"Sample cluster assignments: {clusters[:20]}")

### Finding Deepest Sulci

In [ ]:
# Find the deepest sulcal vertices
# Can be whole brain or within a specific label

deepest_indices = subject.get_deepest_sulci(
    percentage=5,                    # Get deepest 5%
    label_name=None,                 # Whole brain (or specify a label)
    load_label=True,                 # Create a label
    result_label_name='deepest_5pct' # Name for the new label
)

print(f"Found {len(deepest_indices)} vertices in deepest 5% of sulci")
print(f"New label 'deepest_5pct' created")

In [ ]:
# Visualize the deepest sulci
subject.plot(view='lateral')
subject.plot_label('deepest_5pct', view='lateral', face_colors='darkred')
subject.show()

### Label Thresholding

In [ ]:
# Threshold a label based on morphometric measures
# Measures: 'sulc', 'thickness', 'curv', 'label_stat'
# Types: 'absolute', 'percentile'
# Directions: '>', '>=', '<', '<='

# Get thick cortex regions (thickness > 90th percentile)
thick_verts, thick_coords, thick_vals = subject.threshold_label(
    label_name='cortex',
    threshold_type='percentile',
    threshold_direction='>=',
    threshold_value=90,              # 90th percentile
    threshold_measure='thickness',   # Threshold on thickness
    load_label=True,
    new_name='thick_cortex'
)

print(f"Found {len(thick_verts)} vertices with thickness >= 90th percentile")
print(f"Thickness range: {thick_vals.min():.3f} - {thick_vals.max():.3f} mm")

In [ ]:
# Threshold based on sulcal depth (deep sulci)
deep_verts, deep_coords, deep_vals = subject.threshold_label(
    label_name='cortex',
    threshold_type='percentile',
    threshold_direction='>=',
    threshold_value=95,          # Top 5% deepest
    threshold_measure='sulc',    # Sulcal depth values
    load_label=True,
    new_name='deep_sulci_95'
)

print(f"Found {len(deep_verts)} vertices in deepest 5% sulci")

### Label Centroid

In [ ]:
# Calculate the centroid of a label
centroid_coords = subject.label_centroid(
    'cortex',
    load=True  # Creates a 'cortex_centroid' label
)

print(f"Cortex centroid coordinates (RAS): {centroid_coords}")

### Label Boundary Analysis

In [ ]:
# Find the boundary of a label
boundary_indices, boundary_coords = subject.label_boundary(
    'thick_cortex',
    load_label=True  # Creates 'thick_cortex_boundary' label
)

print(f"Boundary vertices: {len(boundary_indices)}")
print(f"Boundary label created: 'thick_cortex_boundary'")

### Combining Labels

In [ ]:
# Combine multiple labels into one
subject.combine_labels(
    label_names=['thick_cortex', 'deep_sulci_95'],
    new_label_name='combined_regions'
)

print(f"Combined label vertices: {len(subject.labels['combined_regions'].vertex_indexes)}")
print(f"\nAll loaded labels: {list(subject.labels.keys())}")

## Summary

This tutorial covered the core functionality of Scalpel:

**Subject & Surface Data:**
- Initialize subjects with `ScalpelSubject()`
- Access vertices, faces, curvature, thickness, and sulcal depth

**Labels:**
- Load from FreeSurfer files: `subject.load_label('name')`
- Create custom labels: `subject.load_label('name', label_idxs=..., label_RAS=...)`
- Combine labels: `subject.combine_labels()`

**Visualization:**
- Plot surfaces: `subject.plot(view='lateral')`
- Plot labels: `subject.plot_label('name', face_colors='red')`

**Measurements:**
- Surface area: `subject.calculate_surface_area()`
- Cortical thickness: `subject.calculate_cortical_thickness()`
- Sulcal depth: `subject.calculate_sulcal_depth()`
- Distance: `subject.calculate_euclidean_distance()`

**Analysis:**
- Gyral clustering: `subject.perform_gyral_clustering()`
- Find deepest sulci: `subject.get_deepest_sulci()`
- Threshold labels: `subject.threshold_label()`
- Label centroid: `subject.label_centroid()`
- Label boundary: `subject.label_boundary()`